## 2025-2026 best so far

## New testing fixing team errors

In [17]:
import json, time, urllib.parse, urllib.request
from datetime import date
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error

# ----------------------------
# SETTINGS
# ----------------------------
TRAIN_CUTOFF = pd.Timestamp("2026-02-07", tz="UTC")     # train on dates < this
PRED_START   = TRAIN_CUTOFF
PRED_END     = pd.Timestamp("2026-03-08", tz="UTC")     # predict Feb 7–Mar 7 (end-exclusive)

FETCH_START_ISO = "2025-11-01"
FETCH_END_ISO   = "2026-03-08"
SLEEP_S = 0.15

PREDICT_FILTER = "acc_vs_acc"  # kept for compatibility; future is driven by CSV anyway

# Pre-cutoff tuning split (must be < TRAIN_CUTOFF)
VAL_START = pd.Timestamp("2026-01-10", tz="UTC")
VAL_END   = TRAIN_CUTOFF

# CSV authority schedule (ACC vs ACC games)
CSV_SCHEDULE_PATH = "data/raw/acc_2025_2026.csv"

# ----------------------------
# TEAM NAME NORMALIZATION
# ----------------------------
# Goal: make ESPN names and CSV names match.
# Add more as you see mismatches in printouts.
NAME_FIX = {
    "NC State": "North Carolina State",
    "N.C. State": "North Carolina State",
    "Miami": "Miami (FL)",
    "Miami FL": "Miami (FL)",
    "Pitt": "Pittsburgh",
    "Cal": "California",
    "Florida St": "Florida State",
    "Florida St.": "Florida State",
    "UNC": "North Carolina",
}

def norm_team(x: str) -> str:
    if pd.isna(x):
        return x
    s = str(x).strip()
    s = " ".join(s.split())
    s = s.replace("&", "and")
    return NAME_FIX.get(s, s)

# ----------------------------
# ESPN fetch helpers
# ----------------------------
BASE_SCOREBOARD = "https://site.api.espn.com/apis/site/v2/sports/basketball/mens-college-basketball/scoreboard"

def get_json(url, params=None):
    if params:
        url = f"{url}?{urllib.parse.urlencode(params)}"
    with urllib.request.urlopen(url) as resp:
        return json.load(resp)

def month_ranges(start_iso, end_iso):
    start = date.fromisoformat(start_iso)
    end = date.fromisoformat(end_iso)
    cur = date(start.year, start.month, 1)
    while cur <= end:
        nxt = date(cur.year + (cur.month == 12), 1 if cur.month == 12 else cur.month + 1, 1)
        month_end = (pd.Timestamp(nxt) - pd.Timedelta(days=1)).date()
        s = max(cur, start)
        e = min(month_end, end)
        yield f"{s:%Y%m%d}-{e:%Y%m%d}"
        cur = nxt

def extract_games(payload):
    rows = []
    for event in payload.get("events", []):
        st = (event.get("season") or {}).get("type")
        if st in (1, "1"):
            continue
        for comp in event.get("competitions", []):
            if st in (3, "3") and not comp.get("conferenceCompetition"):
                continue
            cs = comp.get("competitors", []) or []
            if len(cs) != 2:
                continue
            home = next((c for c in cs if c.get("homeAway") == "home"), None)
            away = next((c for c in cs if c.get("homeAway") == "away"), None)
            if not home or not away:
                continue
            ht = (home.get("team") or {})
            at = (away.get("team") or {})
            rows.append({
                "date": event.get("date"),
                "neutral_site": bool(comp.get("neutralSite")),
                "home_team_short": ht.get("shortDisplayName") or ht.get("displayName"),
                "away_team_short": at.get("shortDisplayName") or at.get("displayName"),
                "home_score": home.get("score"),
                "away_score": away.get("score"),
            })
    return rows

def fetch_season(start_iso, end_iso, sleep_s=0.2):
    rows = []
    for dr in month_ranges(start_iso, end_iso):
        payload = get_json(BASE_SCOREBOARD, params={"dates": dr, "groups": 50, "seasontype": 2, "limit": 1000})
        rows.extend(extract_games(payload))
        time.sleep(sleep_s)
        payload = get_json(BASE_SCOREBOARD, params={"dates": dr, "groups": 50, "seasontype": 3, "limit": 1000})
        rows.extend(extract_games(payload))
        time.sleep(sleep_s)
    return rows

# ----------------------------
# IDs + dedup
# ----------------------------
def make_game_id(d: pd.DataFrame) -> pd.Series:
    day = pd.to_datetime(d["date"], utc=True).dt.strftime("%Y%m%d")
    return (
        day + "|" +
        d["home_team_short"].astype(str) + "|" +
        d["away_team_short"].astype(str) + "|" +
        d["neutral_site"].astype(int).astype(str)
    )

def dedup(df):
    df = df.sort_values("date")
    df = df.drop_duplicates(subset=["date","home_team_short","away_team_short","neutral_site"], keep="first")
    df["game_id"] = make_game_id(df)
    df = df.sort_values(["date","game_id"]).drop_duplicates("game_id", keep="first")
    return df.reset_index(drop=True)

# ----------------------------
# CSV authority schedule -> future games
# ----------------------------
def load_csv_authority_schedule(csv_path: str, start_ts: pd.Timestamp, end_ts: pd.Timestamp) -> pd.DataFrame:
    """
    Expects CSV columns like:
      Date, Visitor/Neutral, Home/Neutral, (optional PTS cols)
    Keeps rows in [start_ts, end_ts).
    """
    s = pd.read_csv(csv_path)
    # Robust rename if the file has those exact names
    if "Date" not in s.columns or "Visitor/Neutral" not in s.columns or "Home/Neutral" not in s.columns:
        raise ValueError(f"CSV missing expected columns. Found: {list(s.columns)}")

    s["date"] = pd.to_datetime(s["Date"], utc=True, errors="coerce")
    s["away_team_short"] = s["Visitor/Neutral"].apply(norm_team)
    s["home_team_short"] = s["Home/Neutral"].apply(norm_team)

    s = s.dropna(subset=["date","home_team_short","away_team_short"]).copy()

    # window filter
    s = s[(s["date"] >= start_ts) & (s["date"] < end_ts)].copy()

    # neutral unknown in CSV -> assume False (most ACC reg season)
    s["neutral_site"] = False

    # create ids for stable merging and for rest-day computation
    s["game_id"] = make_game_id(s)

    # keys for merging to ESPN
    s["date_key"] = s["date"].dt.date
    return s[["game_id","date","date_key","home_team_short","away_team_short","neutral_site"]].reset_index(drop=True)

def merge_csv_with_espn_future(csv_future: pd.DataFrame, espn_future: pd.DataFrame) -> pd.DataFrame:
    """
    Keep ALL csv_future games; enrich with ESPN neutral_site / game_id when available.
    Assumes:
      csv_future has columns: game_id, date, date_key, home_team_short, away_team_short, neutral_site
      espn_future has columns: game_id, date, home_team_short, away_team_short, neutral_site
    """
    e = espn_future.copy()

    # Ensure merge keys exist
    if "date_key" not in e.columns:
        e["date_key"] = e["date"].dt.date

    # Merge: keep all CSV rows
    merged = csv_future.merge(
        e[["game_id", "date_key", "home_team_short", "away_team_short", "neutral_site"]],
        on=["date_key", "home_team_short", "away_team_short"],
        how="left",
        suffixes=("_csv", "_espn"),
        indicator=True
    )

    missing = (merged["_merge"] == "left_only").sum()

    # Prefer ESPN game_id if present, else CSV game_id
    # After merge: csv game_id becomes game_id_csv, ESPN one becomes game_id_espn (because both exist)
    gid_csv = "game_id_csv" if "game_id_csv" in merged.columns else "game_id"
    gid_espn = "game_id_espn" if "game_id_espn" in merged.columns else None

    if gid_espn is not None:
        merged["game_id_final"] = merged[gid_espn].fillna(merged[gid_csv])
    else:
        merged["game_id_final"] = merged[gid_csv]

    # Prefer ESPN neutral_site if present, else CSV neutral_site
    ns_csv = "neutral_site_csv" if "neutral_site_csv" in merged.columns else "neutral_site"
    ns_espn = "neutral_site_espn" if "neutral_site_espn" in merged.columns else None

    if ns_espn is not None:
        merged["neutral_site_final"] = merged[ns_espn].fillna(merged[ns_csv]).astype(bool)
    else:
        merged["neutral_site_final"] = merged[ns_csv].astype(bool)

    out = pd.DataFrame({
        "game_id": merged["game_id_final"],
        "date": merged["date"],  # <-- DATE COMES FROM CSV; no suffix needed
        "home_team_short": merged["home_team_short"],
        "away_team_short": merged["away_team_short"],
        "neutral_site": merged["neutral_site_final"],
    }).sort_values(["date", "game_id"]).reset_index(drop=True)

    print(f"CSV schedule games (authority): {len(csv_future)}")
    print(f"ESPN future games (acc_vs_acc): {len(espn_future)}")
    print(f"Missing from ESPN but kept from CSV: {missing}")
    print(f"FINAL future games to predict: {len(out)}")

    return out


# ----------------------------
# Rest days (schedule-only)
# ----------------------------
def build_rest_from_schedule(df_sched: pd.DataFrame) -> pd.DataFrame:
    d = df_sched.sort_values(["date","game_id"]).reset_index(drop=True).copy()

    home_long = d[["game_id","date","home_team_short"]].rename(columns={"home_team_short":"team"})
    away_long = d[["game_id","date","away_team_short"]].rename(columns={"away_team_short":"team"})
    long = pd.concat([home_long, away_long], ignore_index=True).sort_values(["team","date","game_id"]).reset_index(drop=True)

    long["prev_date"] = long.groupby("team")["date"].shift(1)
    long["rest_days"] = (long["date"] - long["prev_date"]).dt.total_seconds()/86400.0
    long["rest_days"] = long["rest_days"].clip(lower=0, upper=14)
    rest_fallback = float(np.nanmedian(long["rest_days"]))
    long["rest_days"] = long["rest_days"].fillna(rest_fallback)

    home_map = long.merge(
        d[["game_id","home_team_short"]].rename(columns={"home_team_short":"team"}),
        on=["game_id","team"], how="right"
    )[["game_id","rest_days"]].rename(columns={"rest_days":"home_rest_days"})

    away_map = long.merge(
        d[["game_id","away_team_short"]].rename(columns={"away_team_short":"team"}),
        on=["game_id","team"], how="right"
    )[["game_id","rest_days"]].rename(columns={"rest_days":"away_rest_days"})

    m = d[["game_id"]].drop_duplicates("game_id") \
        .merge(home_map, on="game_id", how="left") \
        .merge(away_map, on="game_id", how="left")

    m["rest_diff"] = m["home_rest_days"] - m["away_rest_days"]
    for c in ["home_rest_days","away_rest_days","rest_diff"]:
        m[c] = m[c].fillna(0.0)
    return m

# ----------------------------
# NO-LEAK rolling form/tempo for TRAIN/VAL (as-of each game)
# ----------------------------
def add_no_leak_form_tempo(games: pd.DataFrame, scored_games: pd.DataFrame, form_w: int, min_p: int = 2) -> pd.DataFrame:
    g = games.copy()

    sg = scored_games.dropna(subset=["home_score","away_score"]).copy()
    if "game_id" not in sg.columns:
        raise ValueError("scored_games is missing 'game_id' — ensure dedup() was run before creating df_scored.")
    sg = sg.sort_values(["date","game_id"]).reset_index(drop=True)

    home_long = sg[["game_id","date","home_team_short","home_score","away_score"]].rename(columns={"home_team_short":"team"})
    home_long["pts_for"] = home_long["home_score"]
    home_long["pts_against"] = home_long["away_score"]

    away_long = sg[["game_id","date","away_team_short","home_score","away_score"]].rename(columns={"away_team_short":"team"})
    away_long["pts_for"] = away_long["away_score"]
    away_long["pts_against"] = away_long["home_score"]

    long = pd.concat([home_long, away_long], ignore_index=True)
    long["margin_team"] = long["pts_for"] - long["pts_against"]
    long["tempo_game"]  = (long["pts_for"] + long["pts_against"]) / 2.0
    long = long.sort_values(["team","date","game_id"]).reset_index(drop=True)

    long["form_margin"] = (
        long.groupby("team")["margin_team"]
            .apply(lambda s: s.rolling(form_w, min_periods=min_p).mean().shift(1))
            .reset_index(level=0, drop=True)
    )
    long["tempo_hat"] = (
        long.groupby("team")["tempo_game"]
            .apply(lambda s: s.rolling(form_w, min_periods=min_p).mean().shift(1))
            .reset_index(level=0, drop=True)
    )

    form_fb  = float(np.nanmean(long["margin_team"])) if len(long) else 0.0
    tempo_fb = float(np.nanmean(long["tempo_game"])) if len(long) else 70.0
    long["form_margin"] = long["form_margin"].fillna(form_fb)
    long["tempo_hat"]   = long["tempo_hat"].fillna(tempo_fb)

    home_map = long[["game_id","team","form_margin","tempo_hat"]].rename(
        columns={"team":"home_team_short","form_margin":"home_form_margin","tempo_hat":"home_tempo_hat"}
    )
    away_map = long[["game_id","team","form_margin","tempo_hat"]].rename(
        columns={"team":"away_team_short","form_margin":"away_form_margin","tempo_hat":"away_tempo_hat"}
    )

    g = g.merge(home_map, on=["game_id","home_team_short"], how="left")
    g = g.merge(away_map, on=["game_id","away_team_short"], how="left")

    g["home_form_margin"] = g["home_form_margin"].fillna(form_fb).astype(float)
    g["away_form_margin"] = g["away_form_margin"].fillna(form_fb).astype(float)
    g["form_diff"] = g["home_form_margin"] - g["away_form_margin"]

    g["home_tempo_hat"] = g["home_tempo_hat"].fillna(tempo_fb).astype(float)
    g["away_tempo_hat"] = g["away_tempo_hat"].fillna(tempo_fb).astype(float)
    g["tempo"] = (g["home_tempo_hat"] + g["away_tempo_hat"]) / 2.0

    return g

# ----------------------------
# Freeze team form/tempo at cutoff for FUTURE prediction
# ----------------------------
def freeze_team_form_tempo_at_cutoff(scored_games: pd.DataFrame, cutoff: pd.Timestamp, form_w: int, min_p: int = 2):
    sg = scored_games.dropna(subset=["home_score","away_score"]).copy()
    if "game_id" not in sg.columns:
        raise ValueError("scored_games is missing 'game_id' — ensure dedup() was run before creating df_scored.")
    sg = sg[sg["date"] < cutoff].sort_values(["date","game_id"]).reset_index(drop=True)

    home_long = sg[["game_id","date","home_team_short","home_score","away_score"]].rename(columns={"home_team_short":"team"})
    home_long["pts_for"] = home_long["home_score"]
    home_long["pts_against"] = home_long["away_score"]

    away_long = sg[["game_id","date","away_team_short","home_score","away_score"]].rename(columns={"away_team_short":"team"})
    away_long["pts_for"] = away_long["away_score"]
    away_long["pts_against"] = away_long["home_score"]

    long = pd.concat([home_long, away_long], ignore_index=True)
    long["margin_team"] = long["pts_for"] - long["pts_against"]
    long["tempo_game"]  = (long["pts_for"] + long["pts_against"]) / 2.0
    long = long.sort_values(["team","date","game_id"]).reset_index(drop=True)

    long["form_margin"] = (
        long.groupby("team")["margin_team"]
            .apply(lambda s: s.rolling(form_w, min_periods=min_p).mean().shift(1))
            .reset_index(level=0, drop=True)
    )
    long["tempo_hat"] = (
        long.groupby("team")["tempo_game"]
            .apply(lambda s: s.rolling(form_w, min_periods=min_p).mean().shift(1))
            .reset_index(level=0, drop=True)
    )

    form_fb  = float(np.nanmean(long["margin_team"])) if len(long) else 0.0
    tempo_fb = float(np.nanmean(long["tempo_game"])) if len(long) else 70.0
    long["form_margin"] = long["form_margin"].fillna(form_fb)
    long["tempo_hat"]   = long["tempo_hat"].fillna(tempo_fb)

    last = long.groupby("team").tail(1)
    team_form  = dict(zip(last["team"], last["form_margin"]))
    team_tempo = dict(zip(last["team"], last["tempo_hat"]))
    return team_form, team_tempo, form_fb, tempo_fb

def attach_frozen_form_tempo_future(df_games: pd.DataFrame, team_form: dict, team_tempo: dict,
                                    form_fb: float, tempo_fb: float, form_scale: float = 1.0) -> pd.DataFrame:
    d = df_games.copy()
    d["home_form_margin"] = d["home_team_short"].map(team_form).fillna(form_fb).astype(float) * float(form_scale)
    d["away_form_margin"] = d["away_team_short"].map(team_form).fillna(form_fb).astype(float) * float(form_scale)
    d["form_diff"] = d["home_form_margin"] - d["away_form_margin"]
    d["tempo"] = (
        d["home_team_short"].map(team_tempo).fillna(tempo_fb).astype(float) +
        d["away_team_short"].map(team_tempo).fillna(tempo_fb).astype(float)
    ) / 2.0
    return d

# ============================================================
# 1) Fetch + prep
# ============================================================
df = pd.DataFrame(fetch_season(FETCH_START_ISO, FETCH_END_ISO, sleep_s=SLEEP_S))
df["date"] = pd.to_datetime(df["date"], utc=True, errors="coerce")
df["home_score"] = pd.to_numeric(df["home_score"], errors="coerce")
df["away_score"] = pd.to_numeric(df["away_score"], errors="coerce")
df = df.dropna(subset=["date","home_team_short","away_team_short"]).copy()

# Normalize team names BEFORE dedup / game_id
df["home_team_short"] = df["home_team_short"].apply(norm_team)
df["away_team_short"] = df["away_team_short"].apply(norm_team)

df = dedup(df)

assert "game_id" in df.columns, "game_id missing (dedup not applied?)"
assert df["game_id"].notna().all(), "Some game_id are NaN"

ACC_NEW = {
    "Boston College",
    "California",
    "Clemson",
    "Duke",
    "Florida State",
    "Georgia Tech",
    "Louisville",
    "Miami (FL)",
    "North Carolina State",
    "North Carolina",
    "Notre Dame",
    "Pittsburgh",
    "SMU",
    "Stanford",
    "Syracuse",
    "Virginia",
    "Virginia Tech",
    "Wake Forest",
}

df["home_ACC"] = df["home_team_short"].isin(ACC_NEW)
df["away_ACC"] = df["away_team_short"].isin(ACC_NEW)
df["acc_vs_acc"] = df["home_ACC"] & df["away_ACC"]
df["acc_involved"] = df["home_ACC"] | df["away_ACC"]

df_scored = df.dropna(subset=["home_score","away_score"]).copy()

past = df_scored[df_scored["date"] < TRAIN_CUTOFF].copy().reset_index(drop=True)

# ESPN future window (may be incomplete)
future_window = df[(df["date"] >= PRED_START) & (df["date"] < PRED_END)].copy().reset_index(drop=True)
future_espn_acc = future_window[future_window["acc_vs_acc"]].copy()

# CSV authority schedule
future_csv = load_csv_authority_schedule(CSV_SCHEDULE_PATH, PRED_START, PRED_END)

# FINAL future_pred: CSV-driven, enriched by ESPN when possible
future_pred = merge_csv_with_espn_future(future_csv, future_espn_acc)

# ============================================================
# MANUAL NON-ACC GAMES (challenge-required)
# ============================================================
manual_games = pd.DataFrame([
    {"date": "2026-02-14", "away_team_short": "Baylor",      "home_team_short": "Louisville"},
    {"date": "2026-02-14", "away_team_short": "Ohio State",  "home_team_short": "Virginia"},
    {"date": "2026-02-21", "away_team_short": "Michigan",    "home_team_short": "Duke"},
])

manual_games["date"] = pd.to_datetime(manual_games["date"], utc=True)
manual_games["neutral_site"] = False

# Normalize names to match pipeline
manual_games["home_team_short"] = manual_games["home_team_short"].apply(norm_team)
manual_games["away_team_short"] = manual_games["away_team_short"].apply(norm_team)

# Generate valid game_id so downstream merges work
manual_games["game_id"] = make_game_id(manual_games)

# Append + sort
future_pred = pd.concat(
    [future_pred, manual_games],
    ignore_index=True
).sort_values(["date", "game_id"]).reset_index(drop=True)

print("Future games AFTER manual add:", len(future_pred))


print(f"\nTRAIN scored games (< {TRAIN_CUTOFF.date()}): {len(past)}")
print(f"FUTURE games to predict ({PRED_START.date()}–{(PRED_END - pd.Timedelta(days=1)).date()}, filter={PREDICT_FILTER}): {len(future_pred)}")

# Build rest-days map on an augmented schedule: ESPN schedule + CSV-only games
sched_cols = ["game_id","date","home_team_short","away_team_short","neutral_site"]
df_sched = df[sched_cols].copy()

# Append CSV games not present in ESPN schedule by game_id
missing_ids = set(future_pred["game_id"]) - set(df_sched["game_id"])
csv_only_sched = future_pred[future_pred["game_id"].isin(missing_ids)][sched_cols].copy()

aug_sched = pd.concat([df_sched, csv_only_sched], ignore_index=True)
# Dedup just in case
aug_sched = aug_sched.sort_values(["date","game_id"]).drop_duplicates("game_id", keep="first").reset_index(drop=True)

rest_map_all = build_rest_from_schedule(aug_sched)

# ============================================================
# 2) Train BASIC (pre-cutoff only)
# ============================================================
train_basic = past[past["acc_involved"]].copy().reset_index(drop=True)

teams_basic = sorted(set(train_basic["home_team_short"]) | set(train_basic["away_team_short"]))
idx_basic = {t:i for i,t in enumerate(teams_basic)}
n_basic = len(teams_basic)

def X_spread_basic(d):
    X = np.zeros((len(d), 2*n_basic + 1), dtype=float)
    for i,r in enumerate(d.itertuples(index=False)):
        h = idx_basic.get(r.home_team_short, None)
        a = idx_basic.get(r.away_team_short, None)
        if h is not None: X[i,h] = 1
        if a is not None: X[i,a] = -1
        if h is not None: X[i,n_basic+h] = -1
        if a is not None: X[i,n_basic+a] = 1
        X[i,-1] = 0.0 if bool(r.neutral_site) else 1.0
    return X

Xtr_b = X_spread_basic(train_basic)
ytr_b = (train_basic["home_score"] - train_basic["away_score"]).to_numpy(float)
basic = Ridge(alpha=0.75, fit_intercept=True).fit(Xtr_b, ytr_b)

# ============================================================
# 3) ADV model setup + tuning (pre-cutoff only, NO-LEAK)
# ============================================================
train_full = past[past["acc_involved"]].copy().reset_index(drop=True)

val_raw = train_full[(train_full["date"] >= VAL_START) & (train_full["date"] < VAL_END)].copy().reset_index(drop=True)
trn_raw = train_full[train_full["date"] < VAL_START].copy().reset_index(drop=True)

print(f"\nTuning split (pre-cutoff only): train={len(trn_raw)} | val={len(val_raw)}")

teams_adv = sorted(set(train_full["home_team_short"]) | set(train_full["away_team_short"]))
idx_adv = {t:i for i,t in enumerate(teams_adv)}
n_adv = len(teams_adv)

val_teams = set(val_raw["home_team_short"]) | set(val_raw["away_team_short"])
missing_adv = sorted([t for t in val_teams if t not in idx_adv])
if missing_adv:
    print("⚠️ ADV unseen teams in val (will be treated as zeros):", missing_adv)

def X_points_adv(df_in: pd.DataFrame, home: bool) -> np.ndarray:
    p = 3*n_adv + 1 + 3 + 3 + 1
    X = np.zeros((len(df_in), p), dtype=float)
    for i,r in enumerate(df_in.itertuples(index=False)):
        h = idx_adv.get(r.home_team_short, None)
        a = idx_adv.get(r.away_team_short, None)

        if home:
            if h is not None: X[i, h] = 1.0
            if a is not None: X[i, n_adv + a] = 1.0
            if h is not None: X[i, 2*n_adv + h] = 0.0 if bool(r.neutral_site) else 1.0
        else:
            if a is not None: X[i, a] = 1.0
            if h is not None: X[i, n_adv + h] = 1.0

        X[i, 3*n_adv + 0] = float(r.tempo)

        base = 3*n_adv + 1
        X[i, base + 0] = float(r.home_rest_days)
        X[i, base + 1] = float(r.away_rest_days)
        X[i, base + 2] = float(r.rest_diff)

        base2 = base + 3
        X[i, base2 + 0] = float(r.home_form_margin)
        X[i, base2 + 1] = float(r.away_form_margin)
        X[i, base2 + 2] = float(r.form_diff)

        X[i, -1] = 1.0
    return X

def fit_adv_and_get_preds(train_scored: pd.DataFrame, val_scored: pd.DataFrame,
                          wlo: float, whi: float, alpha_points: float):
    tr = train_scored.copy()
    va = val_scored.copy()

    tr_w = tr.copy()
    for col in ["home_score","away_score"]:
        lo, hi = tr_w[col].quantile([wlo, whi])
        tr_w[col] = tr_w[col].clip(lo, hi)

    Xh_tr = X_points_adv(tr_w, home=True)
    Xa_tr = X_points_adv(tr_w, home=False)
    yh_tr = tr_w["home_score"].to_numpy(float)
    ya_tr = tr_w["away_score"].to_numpy(float)

    m_home = Ridge(alpha=float(alpha_points), fit_intercept=False).fit(Xh_tr, yh_tr)
    m_away = Ridge(alpha=float(alpha_points), fit_intercept=False).fit(Xa_tr, ya_tr)

    pred_tr = m_home.predict(Xh_tr) - m_away.predict(Xa_tr)

    y_spread_tr = (tr_w["home_score"] - tr_w["away_score"]).to_numpy(float)
    Z = np.vstack([np.ones_like(pred_tr), pred_tr]).T
    cal = Ridge(alpha=0.0, fit_intercept=False).fit(Z, y_spread_tr)
    a, b = cal.coef_

    Xh_va = X_points_adv(va, home=True)
    Xa_va = X_points_adv(va, home=False)
    pred_va_raw = m_home.predict(Xh_va) - m_away.predict(Xa_va)

    return float(a), float(b), pred_va_raw

# ----------------------------
# TUNING GRIDS (pre-cutoff only)
# ----------------------------
WINSOR_GRID = [(0.19, 0.81), (0.21, 0.79), (0.23, 0.77)]
ALPHA_GRID  = [0.72, 0.76, 0.80, 0.84]
FORM_W_GRID = [4, 5, 6, 7]

BLEND_W_GRID      = [0.003, 0.005, 0.01, 0.03]
CLIP_GRID         = [14, 16, 18, 20, 22]
SLOPE_SHRINK_GRID = [0.80, 0.85, 0.90]
FORM_SCALE_GRID   = [1.0, 0.85, 0.70, 0.55]

# Cache BASIC preds for val (legal)
X_val_basic = X_spread_basic(val_raw)
pred_val_basic = basic.predict(X_val_basic)
y_val_true = (val_raw["home_score"] - val_raw["away_score"]).to_numpy(float)
val_mae_basic = mean_absolute_error(y_val_true, pred_val_basic)
print(f"\nVAL MAE (BASIC only): {val_mae_basic:.4f}")

best = None
rows = []

# Precompute scored pool used for rolling features in tuning (pre-cutoff only)
scored_pool_pre_cutoff = df_scored[df_scored["date"] < TRAIN_CUTOFF].copy()

for form_w in FORM_W_GRID:
    trn_base = trn_raw.merge(rest_map_all, on="game_id", how="left")
    val_base = val_raw.merge(rest_map_all, on="game_id", how="left")

    trn_feat0 = add_no_leak_form_tempo(trn_base, scored_pool_pre_cutoff, form_w=form_w, min_p=2)
    val_feat0 = add_no_leak_form_tempo(val_base, scored_pool_pre_cutoff, form_w=form_w, min_p=2)

    for form_scale in FORM_SCALE_GRID:
        trn = trn_feat0.copy()
        val = val_feat0.copy()

        trn["home_form_margin"] *= float(form_scale)
        trn["away_form_margin"] *= float(form_scale)
        trn["form_diff"] = trn["home_form_margin"] - trn["away_form_margin"]

        val["home_form_margin"] *= float(form_scale)
        val["away_form_margin"] *= float(form_scale)
        val["form_diff"] = val["home_form_margin"] - val["away_form_margin"]

        for (wlo, whi) in WINSOR_GRID:
            for alpha in ALPHA_GRID:
                cal_a, cal_b, pred_val_adv_raw = fit_adv_and_get_preds(trn, val, wlo=wlo, whi=whi, alpha_points=alpha)

                for s in SLOPE_SHRINK_GRID:
                    b_adj = (1.0 - float(s)) * float(cal_b) + float(s) * 1.0
                    pred_val_adv_cal = float(cal_a) + b_adj * pred_val_adv_raw

                    for w in BLEND_W_GRID:
                        pred_val_mix = float(w) * pred_val_adv_cal + (1.0 - float(w)) * pred_val_basic

                        for C in CLIP_GRID:
                            pred_val_final = np.clip(pred_val_mix, -float(C), float(C))
                            mae = mean_absolute_error(y_val_true, pred_val_final)

                            row = {
                                "form_w": int(form_w),
                                "form_scale": float(form_scale),
                                "winsor": f"{wlo:.3f}-{whi:.3f}",
                                "alpha": float(alpha),
                                "slope_shrink": float(s),
                                "blend_w": float(w),
                                "clip_C": float(C),
                                "val_mae": float(mae),
                                "cal_a": float(cal_a),
                                "cal_b": float(cal_b),
                                "b_adj": float(b_adj),
                            }
                            rows.append(row)
                            if best is None or mae < best["val_mae"]:
                                best = row

tune = pd.DataFrame(rows).sort_values("val_mae").reset_index(drop=True)
print("\nTop 20 configs by VAL MAE (pre-cutoff only):")
print(tune.head(20).to_string(index=False))
print("\n✅ Selected ONE-SHOT config:", best)
print(f"BEST VAL MAE (final post-processed): {best['val_mae']:.4f}")

# Recompute ADV-only calibrated val MAE (no blend/clip) for the selected best config
best_form_w = int(best["form_w"])
best_form_scale = float(best["form_scale"])
best_wlo, best_whi = map(float, best["winsor"].split("-"))
best_alpha = float(best["alpha"])
best_s = float(best["slope_shrink"])

trn_base = trn_raw.merge(rest_map_all, on="game_id", how="left")
val_base = val_raw.merge(rest_map_all, on="game_id", how="left")
trn_feat0 = add_no_leak_form_tempo(trn_base, scored_pool_pre_cutoff, form_w=best_form_w, min_p=2)
val_feat0 = add_no_leak_form_tempo(val_base, scored_pool_pre_cutoff, form_w=best_form_w, min_p=2)

trn_feat0["home_form_margin"] *= best_form_scale
trn_feat0["away_form_margin"] *= best_form_scale
trn_feat0["form_diff"] = trn_feat0["home_form_margin"] - trn_feat0["away_form_margin"]

val_feat0["home_form_margin"] *= best_form_scale
val_feat0["away_form_margin"] *= best_form_scale
val_feat0["form_diff"] = val_feat0["home_form_margin"] - val_feat0["away_form_margin"]

cal_a, cal_b, pred_val_adv_raw = fit_adv_and_get_preds(trn_feat0, val_feat0, wlo=best_wlo, whi=best_whi, alpha_points=best_alpha)
b_adj = (1.0 - best_s) * float(cal_b) + float(best_s) * 1.0
pred_val_adv_cal = float(cal_a) + float(b_adj) * pred_val_adv_raw
val_mae_adv_cal = mean_absolute_error(y_val_true, pred_val_adv_cal)
print(f"VAL MAE (ADV calibrated only; no blend/clip): {val_mae_adv_cal:.4f}")

# ============================================================
# 4) Refit ADV once on ALL pre-cutoff ACC-involved with best config (NO-LEAK)
# ============================================================
best_blend_w = float(best["blend_w"])
best_clip_C = float(best["clip_C"])

train_adv = train_full.merge(rest_map_all, on="game_id", how="left")
train_adv = add_no_leak_form_tempo(train_adv, scored_pool_pre_cutoff, form_w=best_form_w, min_p=2)
train_adv["home_form_margin"] *= best_form_scale
train_adv["away_form_margin"] *= best_form_scale
train_adv["form_diff"] = train_adv["home_form_margin"] - train_adv["away_form_margin"]

# Winsorize TRAIN scores only
train_adv_w = train_adv.copy()
for col in ["home_score","away_score"]:
    lo, hi = train_adv_w[col].quantile([best_wlo, best_whi])
    train_adv_w[col] = train_adv_w[col].clip(lo, hi)

Xh_tr = X_points_adv(train_adv_w, home=True)
Xa_tr = X_points_adv(train_adv_w, home=False)
yh_tr = train_adv_w["home_score"].to_numpy(float)
ya_tr = train_adv_w["away_score"].to_numpy(float)

m_home = Ridge(alpha=float(best_alpha), fit_intercept=False).fit(Xh_tr, yh_tr)
m_away = Ridge(alpha=float(best_alpha), fit_intercept=False).fit(Xa_tr, ya_tr)

pred_tr_raw = m_home.predict(Xh_tr) - m_away.predict(Xa_tr)

# Base calibration (pre-cutoff only)
y_spread_tr = (train_adv_w["home_score"] - train_adv_w["away_score"]).to_numpy(float)
Z = np.vstack([np.ones_like(pred_tr_raw), pred_tr_raw]).T
cal = Ridge(alpha=0.0, fit_intercept=False).fit(Z, y_spread_tr)
cal_a, cal_b = cal.coef_
b_adj = (1.0 - best_s) * float(cal_b) + float(best_s) * 1.0

print(f"\nFinal calibration (pre-cutoff): a={float(cal_a):.4f}, b={float(cal_b):.4f}, b_adj={b_adj:.4f}")
print(f"Post-process: blend_w={best_blend_w:.2f}, clip_C={best_clip_C:.0f}, form_scale={best_form_scale:.2f}")

# ============================================================
# 5) Predict Feb 7–Mar 7 (one-shot, frozen features at TRAIN_CUTOFF)
# ============================================================
X_future_basic = X_spread_basic(future_pred)
pred_basic_future = basic.predict(X_future_basic)

team_form, team_tempo, form_fb, tempo_fb = freeze_team_form_tempo_at_cutoff(
    scored_games=scored_pool_pre_cutoff, cutoff=TRAIN_CUTOFF, form_w=best_form_w, min_p=2
)

future_feat = future_pred.merge(rest_map_all, on="game_id", how="left")

# If rest columns are still missing for any reason, fill to safe defaults
for c in ["home_rest_days","away_rest_days","rest_diff"]:
    if c not in future_feat.columns:
        future_feat[c] = 0.0
    future_feat[c] = future_feat[c].fillna(0.0)

future_feat = attach_frozen_form_tempo_future(
    future_feat, team_form, team_tempo, form_fb, tempo_fb, form_scale=best_form_scale
)

Xh_fu = X_points_adv(future_feat, home=True)
Xa_fu = X_points_adv(future_feat, home=False)
pred_adv_raw_future = m_home.predict(Xh_fu) - m_away.predict(Xa_fu)
pred_adv_cal_future = float(cal_a) + float(b_adj) * pred_adv_raw_future

pred_final = best_blend_w * pred_adv_cal_future + (1.0 - best_blend_w) * pred_basic_future
pred_final = np.clip(pred_final, -best_clip_C, best_clip_C)

submit = future_feat[["date","home_team_short","away_team_short","neutral_site"]].copy()
submit["pred_basic_spread"] = pred_basic_future
submit["pred_adv_spread"] = pred_adv_cal_future
submit["pred_final_spread"] = pred_final
submit = submit.sort_values("date").reset_index(drop=True)

print("\nSUBMISSION PREVIEW (all 78 predicted games):")
print(submit.head(78).to_string(index=False))

print("\nCounts check:")
print("Train games:", len(past))
print("Validation games:", len(val_raw))
print("Future games:", len(future_pred))

print("Any NaN preds?", submit["pred_final_spread"].isna().any())
print("Date range:", submit["date"].min(), "→", submit["date"].max())

# Save if needed:
# submit.to_csv("one_shot_predictions_no_leak_feb7_mar7.csv", index=False)
# print("\nSaved: one_shot_predictions_no_leak_feb7_mar7.csv")


CSV schedule games (authority): 75
ESPN future games (acc_vs_acc): 59
Missing from ESPN but kept from CSV: 46
FINAL future games to predict: 75
Future games AFTER manual add: 78

TRAIN scored games (< 2026-02-07): 2188
FUTURE games to predict (2026-02-07–2026-03-07, filter=acc_vs_acc): 78

Tuning split (pre-cutoff only): train=109 | val=37

VAL MAE (BASIC only): 7.4637


/var/folders/4s/n8b628fx4s54lh603q69xk7h0000gn/T/ipykernel_45685/1609862051.py:287: FutureWarning: Not prepending group keys to the result index of transform-like apply. In the future, the group keys will be included in the index, regardless of whether the applied function returns a like-indexed object.
To preserve the previous behavior, use

	>>> .groupby(..., group_keys=False)

To adopt the future behavior and silence this warning, use 

	>>> .groupby(..., group_keys=True)
  .apply(lambda s: s.rolling(form_w, min_periods=min_p).mean().shift(1))
/var/folders/4s/n8b628fx4s54lh603q69xk7h0000gn/T/ipykernel_45685/1609862051.py:292: FutureWarning: Not prepending group keys to the result index of transform-like apply. In the future, the group keys will be included in the index, regardless of whether the applied function returns a like-indexed object.
To preserve the previous behavior, use

	>>> .groupby(..., group_keys=False)

To adopt the future behavior and silence this warning, use 

	>>


Top 20 configs by VAL MAE (pre-cutoff only):
 form_w  form_scale      winsor  alpha  slope_shrink  blend_w  clip_C  val_mae     cal_a    cal_b    b_adj
      5        1.00 0.230-0.770   0.84          0.90    0.003    20.0 7.381367 -3.068568 1.356610 1.035661
      5        0.85 0.230-0.770   0.84          0.90    0.003    20.0 7.381367 -3.068633 1.356617 1.035662
      5        0.70 0.230-0.770   0.84          0.90    0.003    20.0 7.381367 -3.068744 1.356630 1.035663
      5        0.55 0.230-0.770   0.84          0.90    0.003    20.0 7.381368 -3.068956 1.356655 1.035665
      5        1.00 0.230-0.770   0.80          0.90    0.003    20.0 7.381427 -2.970863 1.345258 1.034526
      5        0.85 0.230-0.770   0.80          0.90    0.003    20.0 7.381428 -2.970924 1.345265 1.034527
      5        0.70 0.230-0.770   0.80          0.90    0.003    20.0 7.381428 -2.971027 1.345277 1.034528
      5        0.55 0.230-0.770   0.80          0.90    0.003    20.0 7.381428 -2.971226 1.345300 

/var/folders/4s/n8b628fx4s54lh603q69xk7h0000gn/T/ipykernel_45685/1609862051.py:287: FutureWarning: Not prepending group keys to the result index of transform-like apply. In the future, the group keys will be included in the index, regardless of whether the applied function returns a like-indexed object.
To preserve the previous behavior, use

	>>> .groupby(..., group_keys=False)

To adopt the future behavior and silence this warning, use 

	>>> .groupby(..., group_keys=True)
  .apply(lambda s: s.rolling(form_w, min_periods=min_p).mean().shift(1))
/var/folders/4s/n8b628fx4s54lh603q69xk7h0000gn/T/ipykernel_45685/1609862051.py:292: FutureWarning: Not prepending group keys to the result index of transform-like apply. In the future, the group keys will be included in the index, regardless of whether the applied function returns a like-indexed object.
To preserve the previous behavior, use

	>>> .groupby(..., group_keys=False)

To adopt the future behavior and silence this warning, use 

	>>

VAL MAE (ADV calibrated only; no blend/clip): 10.8317

Final calibration (pre-cutoff): a=-1.8619, b=1.3023, b_adj=1.0302
Post-process: blend_w=0.00, clip_C=20, form_scale=1.00


/var/folders/4s/n8b628fx4s54lh603q69xk7h0000gn/T/ipykernel_45685/1609862051.py:287: FutureWarning: Not prepending group keys to the result index of transform-like apply. In the future, the group keys will be included in the index, regardless of whether the applied function returns a like-indexed object.
To preserve the previous behavior, use

	>>> .groupby(..., group_keys=False)

To adopt the future behavior and silence this warning, use 

	>>> .groupby(..., group_keys=True)
  .apply(lambda s: s.rolling(form_w, min_periods=min_p).mean().shift(1))
/var/folders/4s/n8b628fx4s54lh603q69xk7h0000gn/T/ipykernel_45685/1609862051.py:292: FutureWarning: Not prepending group keys to the result index of transform-like apply. In the future, the group keys will be included in the index, regardless of whether the applied function returns a like-indexed object.
To preserve the previous behavior, use

	>>> .groupby(..., group_keys=False)

To adopt the future behavior and silence this warning, use 

	>>


SUBMISSION PREVIEW (all 78 predicted games):
                     date      home_team_short      away_team_short  neutral_site  pred_basic_spread  pred_adv_spread  pred_final_spread
2026-02-07 00:00:00+00:00       Boston College           Miami (FL)         False          -4.970286         2.217681          -4.948722
2026-02-07 00:00:00+00:00           California              Clemson         False          -2.309569         2.153369          -2.296180
2026-02-07 00:00:00+00:00 North Carolina State        Virginia Tech         False           7.770702         7.088289           7.768655
2026-02-07 00:00:00+00:00       North Carolina                 Duke         False          -6.032813         6.465424          -5.995319
2026-02-07 00:00:00+00:00           Notre Dame        Florida State         False           4.434347        11.324191           4.455016
2026-02-07 00:00:00+00:00           Pittsburgh   Southern Methodist         False           3.363399        -0.313042           3.35

/var/folders/4s/n8b628fx4s54lh603q69xk7h0000gn/T/ipykernel_45685/1609862051.py:350: FutureWarning: Not prepending group keys to the result index of transform-like apply. In the future, the group keys will be included in the index, regardless of whether the applied function returns a like-indexed object.
To preserve the previous behavior, use

	>>> .groupby(..., group_keys=False)

To adopt the future behavior and silence this warning, use 

	>>> .groupby(..., group_keys=True)
  .apply(lambda s: s.rolling(form_w, min_periods=min_p).mean().shift(1))
